# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mishellscripts/flyrank/blob/main/work/notebooks/w07_action_playbook.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Build model from ML-05

In [2]:
# --- Setup ---
import duckdb
from google.colab import userdata
import pandas as pd
import numpy as np

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
url = "hf://datasets/FlyRank/internship-warehouse"
rel = f"{url}/fact_content_daily_performance/**/*.parquet"
print("Connected!")

Connected!


In [3]:
T = "2026-02-01"  # validated cutoff -- 39 eligible clients

WINDOW_DAYS = 30
LABEL_GAP_DAYS = 1
LABEL_WINDOW_DAYS = 30

# --- Feature set, as of T only ---
content_type_query = f"""
    SELECT content_hash_id, keyword_char_count, url_char_count, content_type,
        search_volume, competition, main_intent, category_count, model_used, char_count,
        DATEDIFF('day', content_updated_date, DATE '{T}') AS days_since_last_update,
        DATEDIFF('day', content_created_date, DATE '{T}') AS days_since_created
    FROM read_parquet('{url}/dim_content.parquet')
"""
content_dim = con.sql(content_type_query).df()

feature_query = f"""
    SELECT client_hash_id, content_hash_id,
        AVG(gsc_avg_position) AS gsc_avg_position,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks
    FROM read_parquet('{rel}')
    WHERE report_date <= DATE '{T}'
    GROUP BY client_hash_id, content_hash_id
"""
X_raw = con.sql(feature_query).df().merge(content_dim, on="content_hash_id", how="left")

# --- Gate: is_declining at T ---
gate_query = f"""
    WITH windowed AS (
        SELECT client_hash_id, content_hash_id,
            SUM(CASE WHEN report_date BETWEEN (DATE '{T}' - INTERVAL {WINDOW_DAYS - 1} DAY) AND DATE '{T}'
                     THEN gsc_impressions ELSE 0 END) AS impr_last30,
            SUM(CASE WHEN report_date BETWEEN (DATE '{T}' - INTERVAL {2*WINDOW_DAYS - 1} DAY)
                                           AND (DATE '{T}' - INTERVAL {WINDOW_DAYS} DAY)
                     THEN gsc_impressions ELSE 0 END) AS impr_prev30
        FROM read_parquet('{rel}')
        WHERE report_date <= DATE '{T}'
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT client_hash_id, content_hash_id,
        CASE WHEN (impr_last30 - impr_prev30) / NULLIF(impr_prev30, 0) * 100 < -20
             THEN 1 ELSE 0 END AS is_declining_at_T
    FROM windowed
"""
gate_df = con.sql(gate_query).df()

# --- Recovery label: strictly after T ---
future_query = f"""
    WITH future AS (
        SELECT client_hash_id, content_hash_id,
            SUM(CASE WHEN report_date BETWEEN (DATE '{T}' + INTERVAL {LABEL_GAP_DAYS + LABEL_WINDOW_DAYS} DAY)
                                           AND (DATE '{T}' + INTERVAL {LABEL_GAP_DAYS + 2*LABEL_WINDOW_DAYS - 1} DAY)
                     THEN gsc_impressions ELSE 0 END) AS impr_T1,
            SUM(CASE WHEN report_date BETWEEN (DATE '{T}' + INTERVAL {LABEL_GAP_DAYS} DAY)
                                           AND (DATE '{T}' + INTERVAL {LABEL_GAP_DAYS + LABEL_WINDOW_DAYS - 1} DAY)
                     THEN gsc_impressions ELSE 0 END) AS impr_baseline
        FROM read_parquet('{rel}')
        WHERE report_date > DATE '{T}'
          AND report_date <= (DATE '{T}' + INTERVAL {LABEL_GAP_DAYS + 2*LABEL_WINDOW_DAYS - 1} DAY)
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT client_hash_id, content_hash_id,
        CASE WHEN impr_T1 > impr_baseline THEN 1 ELSE 0 END AS recovered_by_T1
    FROM future
"""
recovery_df = con.sql(future_query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [4]:
# --- Impute missing data ---
X_raw["has_keyword_data"] = X_raw["search_volume"].notna() & X_raw["competition"].notna()
X_raw["char_count_imputed"] = X_raw["char_count"].isna()
X_raw["ai_generated"] = X_raw["model_used"].notna()

X_raw["gsc_avg_position"] = X_raw["gsc_avg_position"].fillna(0)
X_raw["search_volume"] = X_raw["search_volume"].fillna(0)
X_raw["competition"] = X_raw["competition"].fillna(0)
X_raw["main_intent"] = X_raw["main_intent"].fillna("no_keyword")
X_raw["model_used"] = X_raw["model_used"].fillna("human")
X_raw["char_count"] = X_raw["char_count"].fillna(X_raw["char_count"].median())

# --- Assemble data, y, groups ---
declining_ids = gate_df.loc[gate_df["is_declining_at_T"] == 1, ["client_hash_id", "content_hash_id"]]

data = (X_raw.merge(declining_ids, on=["client_hash_id", "content_hash_id"], how="inner")
             .merge(recovery_df[["client_hash_id", "content_hash_id", "recovered_by_T1"]],
                     on=["client_hash_id", "content_hash_id"], how="inner"))

y = data.pop("recovered_by_T1")
groups = data["client_hash_id"]
categorical_text_cols = ["content_type", "main_intent", "model_used"]

# --- Build R2: drop confound cluster, add ctr, encode without model_used ---
drop_r2 = ["char_count", "char_count_imputed", "keyword_char_count", "gsc_clicks", "model_used", "ai_generated"]
categorical_r2 = [c for c in categorical_text_cols if c != "model_used"]

X_r2 = pd.get_dummies(
    data.drop(columns=["client_hash_id", "content_hash_id"] +
              [c for c in drop_r2 if c in data.columns]),
    columns=categorical_r2
)

X_r2["ctr"] = data["gsc_clicks"] / data["gsc_impressions"].replace(0, np.nan)
X_r2["ctr"] = X_r2["ctr"].fillna(0)

print(f"X_r2: {X_r2.shape[0]} rows, {X_r2.shape[1]} features, {groups.nunique()} clients")
print(f"y positive rate: {y.mean():.3f}")

X_r2: 34402 rows, 17 features, 38 clients
y positive rate: 0.402


In [5]:
from sklearn.ensemble import RandomForestClassifier

rf_final = RandomForestClassifier(
    max_depth=4,
    random_state=42,
    class_weight="balanced"
)
rf_final.fit(X_r2, y)

print(f"Trained on {len(X_r2)} rows, {X_r2.shape[1]} features")

Trained on 34402 rows, 17 features


In [6]:
# --- P(recovery) from the trained model, for every declining page ---
p_recovery = rf_final.predict_proba(X_r2)[:, 1]

# --- impact_at_risk: how much is at stake, using decline magnitude x scale ---
# trend_pct at T (the gate's own computation) + impressions, both real/observed, no leakage
# rebuild trend_pct_at_T from the same windowed query used for the gate
trend_query = f"""
    SELECT client_hash_id, content_hash_id,
        (impr_last30 - impr_prev30) / NULLIF(impr_prev30, 0) * 100 AS trend_pct_at_T
    FROM (
        SELECT client_hash_id, content_hash_id,
            SUM(CASE WHEN report_date BETWEEN (DATE '{T}' - INTERVAL {WINDOW_DAYS - 1} DAY) AND DATE '{T}'
                     THEN gsc_impressions ELSE 0 END) AS impr_last30,
            SUM(CASE WHEN report_date BETWEEN (DATE '{T}' - INTERVAL {2*WINDOW_DAYS - 1} DAY)
                                           AND (DATE '{T}' - INTERVAL {WINDOW_DAYS} DAY)
                     THEN gsc_impressions ELSE 0 END) AS impr_prev30
        FROM read_parquet('{rel}')
        WHERE report_date <= DATE '{T}'
        GROUP BY client_hash_id, content_hash_id
    )
"""
trend_df = con.sql(trend_query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [13]:
# join trend_pct onto your declining-page population (same rows as X_r2/y, in order)
scored = data[["client_hash_id", "content_hash_id"]].merge(
    trend_df, on=["client_hash_id", "content_hash_id"], how="left"
)

# impact_at_risk = normalized impressions x normalized decline severity
log_impr = np.log1p(data["gsc_impressions"])
impact_at_risk = (log_impr / log_impr.max()) * (scored["trend_pct_at_T"].abs() / 100)

# --- final priority score ---
scored["p_recovery"] = p_recovery
scored["impact_at_risk"] = impact_at_risk
scored["priority_score"] = scored["impact_at_risk"] * (1 - scored["p_recovery"])

# top quartile impact, bottom quartile recovery -- both genuine minorities, not halves
high_impact = scored["impact_at_risk"] >= scored["impact_at_risk"].quantile(0.75)
low_recovery = scored["p_recovery"] <= scored["p_recovery"].quantile(0.25)

def reason_code(impact_h, recov_l):
    if impact_h and recov_l:
        return "high_impact_low_recovery"
    if impact_h and not recov_l:
        return "high_impact_likely_recovers"
    if not impact_h and recov_l:
        return "low_impact_low_recovery"
    return "low_impact_likely_recovers"

scored["reason_code"] = [
    reason_code(h, r) for h, r in zip(high_impact, low_recovery)
]

scored = scored.sort_values("priority_score", ascending=False).reset_index()
scored.index += 1
print(scored.head(20)[["client_hash_id", "content_hash_id", "priority_score", "reason_code", "impact_at_risk", "p_recovery"]].to_string())

             client_hash_id           content_hash_id  priority_score                  reason_code  impact_at_risk  p_recovery
1   client_1d09b519bdde7c7a  content_5519ab199297bdb2        0.532372     high_impact_low_recovery        0.720535    0.261144
2   client_1d09b519bdde7c7a  content_c9a819d1b9ad7420        0.464083     high_impact_low_recovery        0.627437    0.260353
3   client_1d09b519bdde7c7a  content_972690231ea8c69f        0.460388     high_impact_low_recovery        0.598538    0.230812
4   client_c182d11e4862a37d  content_eb77135ddda07ea8        0.456406     high_impact_low_recovery        0.567226    0.195372
5   client_861cdcccf8049915  content_eea3b2434e98edf3        0.454795     high_impact_low_recovery        0.667077    0.318227
6   client_23a62021009f63c4  content_469512b149928d3e        0.454790  high_impact_likely_recovers        0.859845    0.471079
7   client_2e65897d94f60220  content_6945124cd32196d5        0.449279     high_impact_low_recovery        0.514

Every page above is already known to be declining (impressions fell more
than 20% over 30 days, as of the cutoff date). This queue ranks those pages
by two things combined: how much traffic is at stake (`impact_at_risk`)
and how unlikely the page is to recover on its own (1 - a trained model's
probability of recovery (`p_recovery`)).

`priority_score` and `reason_code` can disagree at the margins. A page with very high `impact_at_risk` can rank near the top by score even
if its `p_recovery` sits just above the low_recovery threshold (see rows 6
and 10 above, both high_impact_likely_recovers but still top-10 by score).
All columns are shown together so a reviewer can see whether
a page's rank is driven mostly by stakes or mostly by low recovery
confidence.

In [12]:
scored["reason_code"].value_counts()

,count
reason_code,
low_impact_likely_recovers,18287
low_impact_low_recovery,7514
high_impact_likely_recovers,7505
high_impact_low_recovery,1096


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

An editor or content lead can use the priority scores to decide what to
review first. The list answers "which
of my known problems deserve attention first" rather than "which currently-healthy
pages are about to become problems". The key distinction is problem mitigation rather than prevention.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Before acting on any flagged page, a real person should check:

1. **Is the decline real, or a deindexing/retirement case?** A page can show
   -100% trend_pct because it's genuinely declining, or because it was
   deindexed, merged into another page, or intentionally retired. These
   look identical in impact_at_risk and the recovery model has no way to
   tell them apart. Check whether the page still resolves and is meant to
   be live before treating a "high urgency" flag as a content problem.

2. **Does the client this page belongs to resemble the 39 eligible
   clients the model was built on?** The model was trained on a small data set of 39 clients. A very new client, or one with a
   thin history, is being scored by a model that has effectively never
   seen a client like them. Treat scores for such clients as much lower
   confidence, or exclude them from the queue until more history exists.

3. **Not only the reason code, but also the continuous priority score value, `impact_at_risk`, and `p_recovery`**

The model was trained to identify pages worth a
  human review, not to make the review decision itself. Never auto-remove, auto-merge, or auto-deprioritize a page based on
  this queue alone.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Signals should be checked on a recurring basis (e.g. monthly, alongside each
new scoring run):
- `p_recovery` drifting from original value
- Correlation audit as done in ML-05 to confirm dropped/added features
- Coefficient values of features changing and becoming unstable
- New `content_type` or `main_intent` category appearing


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [14]:
import os
import matplotlib.pyplot as plt

os.makedirs("work/outputs", exist_ok=True)

# --- 1. the full ranked queue, every declining page, with all fields ---
scored.to_csv("work/outputs/ml10_action_queue.csv", index=False)
print(f"Wrote {len(scored)} rows to work/outputs/ml10_action_queue.csv")

# --- 2. summary stats table, for a table in the paper ---
summary_stats = pd.DataFrame({
    "metric": [
        "total_declining_pages", "eligible_clients", "cutoff_date_T",
        "high_impact_low_recovery_count", "cross_val_auc_range",
        "cross_val_precision50_lift_range",
    ],
    "value": [
        len(scored),
        groups.nunique(),
        T,
        int((scored["reason_code"] == "high_impact_low_recovery").sum()),
        "0.60-0.68",
        "0.03-0.33",
    ]
})
summary_stats.to_csv("work/outputs/ml10_summary_stats.csv", index=False)

# --- 3. reason code distribution figure ---
fig, ax = plt.subplots(figsize=(8, 5))
scored["reason_code"].value_counts().plot(kind="barh", ax=ax, color="#4C72B0")
ax.set_xlabel("Number of pages")
ax.set_title("Declining pages by reason code")
plt.tight_layout()
plt.savefig("work/outputs/ml10_reason_code_distribution.png", dpi=150)
plt.close()

# --- 4. top-20 preview, for a table directly in the paper text ---
scored.head(20).to_csv("work/outputs/ml10_top20_preview.csv", index=False)

print("\nAll files written to work/outputs/:")
for f in ["ml10_action_queue.csv", "ml10_summary_stats.csv",
          "ml10_reason_code_distribution.png", "ml10_top20_preview.csv"]:
    print(f"  - {f}")

Wrote 34402 rows to work/outputs/ml10_action_queue.csv

All files written to work/outputs/:
  - ml10_action_queue.csv
  - ml10_summary_stats.csv
  - ml10_reason_code_distribution.png
  - ml10_top20_preview.csv


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.